# Sentinel-3 OLCI L2 ocean colour

Fetch a Sentinel-3 OLCI Level 2 Water Full Resolution (WFR) product — a NetCDF product `pyramids` can read today.

This notebook performs a **real** EUMETSAT Data Store download when `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` are set (see the [authentication guide](../../reference/eumetsat/authentication.md)). Without credentials it prints a `skipped live query` line and still runs to the end.

## Setup

Import the unified `EarthLens` entry point and `Path`, then create the output directory the download will write into.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('eumetsat_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the OLCI L2 WFR product

Build the request first — source, dataset, product variable, a one-day window, and a bounding box over the western Mediterranean. Keeping the `EarthLens(...)` construction on its own line makes each request parameter easy to read and tweak.

In [ ]:
request = EarthLens(
    data_source='eumetsat',
    dataset='s3-olci-l2-wfr',
    variables=['OL_2_WFR'],
    start='2024-06-01',
    end='2024-06-01',
    aoi=[0.0, 40.0, 5.0, 45.0],
    path=str(OUT_DIR),
)

`download()` is wrapped in a `try`/`except` so the notebook still runs top-to-bottom without credentials: if `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` are missing — or the Data Store has a transient hiccup — it prints a `skipped live query` line instead of raising.

In [ ]:
paths = None
try:
    paths = request.download(progress_bar=False)
    print(len(paths), 'product(s):', [Path(p).name for p in paths])
except Exception as exc:
    # No EUMETSAT_CONSUMER_KEY/SECRET, or a transient Data Store hiccup:
    # the notebook still runs top-to-bottom and reports the skip.
    print(f'skipped live query: {type(exc).__name__}: {exc}')

OLCI L2 WFR carries chlorophyll and other ocean-colour geophysical products as NetCDF — open the downloaded `.nc` with xarray or `pyramids.netcdf.NetCDF.read_file`.